# Kaggriculture | Soil V25 Exact

An independent public high-score route used as a paired submission beside Tran Adaptive Edge.

## Method: Independent Soil Route

This notebook intentionally does not combine Tran and Soil. It preserves the Soil V25 public policy as one coherent system: its 720-step trace, weed-aware pasture repair, terminal conversion and clone-aware market behavior.

Submitting it separately gives a useful result: both routes face the same contemporary opponent pool, so their scores can be compared without guessing which overlay caused the difference.

In [ ]:
from pathlib import Path
import ast
import hashlib
import io
import json
import py_compile
import tarfile

ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
MAIN_PATH = ROOT / 'main.py'
ARCHIVE_PATH = ROOT / 'submission.tar.gz'
print({'working_directory': str(ROOT)})


## Agent

In [ ]:
AGENT_SOURCE = r'''"""Soil V25 exact replay policy."""
import base64
import copy
import gzip
import json

TRACE_B64 = 'H4sIAAAAAAAC/+1dTW8cWXL8Lzz3gd1NUqRvXKl3RliuJFDSNtYDYjCA1zBgrA9j3wz/d2vE7uqqysjIyHyvSErQjWiSXe+7MuNFRP7yv2f//tvv//zH72f/8svZh9uPH88eVmf/8dt//dt/f/ngy4///O33//zH/3z5+Zezn9/e7778dvTDnz7//dfbd2//ent3tjp7/X5/tlofP/6427358uFfd3fv352trmYf73/e3X46fPxxd3d3+mhtP9o+PPzfatzMP31+e/fm1y+N/fT5a0OG9v5ytt99/PS1ae/e33/6+exh2oM/WvDh/v2bz68/zRqR6NLh3y7nrfrw9vVfPn8Y/deoWYfWrMY/PTZVaODGa9e8AXe3r3eH306ePh+vTq34+PNu9wG14zgQp/8cNebj+8+HR9tmzT4RWrU9Td6sFX/+Y7YmT55O0GY6Drvbw8JBAwKWKGzNGq1nb3SG4Ru38fWtmaHho8cfpBU9e+bxy+CK/LJs3n0atur02Y8HgvDItfDI6fr8MtK3n3b3wgPV0d/gs8fdJ48TgId/OEdOy7Wlaet4dY4eRCYEHmv6AM1acegu2iPjMUUTdWgua8hGb8hhuOH6HKZi+EFelRtlPx5G+/H3kzYcJgnNiDtHfEDm3T4MLFwGp1Ef/aSe1tt4hO24zn4YP0YftPkuV8Zu9nWl/tmhHKYPjuTjT7npMmNozwv6HoFduUisyNMWMGM4PJgOZhBZgVE8LYph7OYNdEZT6y14UYDDl2+BYN74ap11xX5iT2Z9nV4oIwyWp+k524wk3pKauDKxffi24nE2eA/wB/CQ8RQBvb+7273+9Oufd/ef3t69/dfJkkgF/dtcC8DbiL6Ti90L+tn6oFFMSR8AY81cl/D3zw5HqUs0dn1z//6DsAQO63HUGntIgDV+OvAe82J7GHaKjGeRDX8V10NM0CHpKJ998fFs4h3IxyymB+hBtgujvRk2nURBUssrQ5xp32GD4LTANhAG4drYzqEe2PTTw8GTwi88RSR9vo82UHhNCg08PeLxd8/bvuzcV75wGINCh+34Jb5NDlrg4d09ZJFDIwdjGCWa79/foah+wRBHzbtvigGYEv7ksMRFAyQpg+0bKfUIA7WZz+XpsyeuIwRAy5QzEU0MZrW+aX2gpLn1KYykR2BGMsDm2OwUfdthaogkaJQ2PNNEQcpLwoZrp1EBvZmj+an1hL7Zbovwi20EN3zdKVmxw2O/13/xVQM8EO4oKHeHMKrX96I9UfjiYSL8wJTj/eMpEQb+lG3lnveSvni2UqWvzYZ446gLjvTzRX3PjVetX0zktrLT9SOYSwNgCwZz4DXTLZizmEanGM5FX/q1+clCOHp78c1FcD6OFdyijY/u64dEZAfWQqfIzsVxi+CcFPmg0G7REK4LkiN15EcI9yOEY8O6gqHd9x/CqXjcGuOLh38/YIsXC0Z8qUFdHrXrd6daifkSl5Kl4I9Ow3KY3s+393+Lfu6EkqGjK04yplD6FtNtt/E54XVu4AFpdJIwUvE5IwjkHJ5++mhonDsWHz/d3+7/tLu//7ulNersJJQAHJs6eoJDsgN/6La3QK5De8BGtblFyuhL428aLYTdR4/2E64MHjY73zfvU6JzhMFHHms/ASORG1O73DlrkbJGwyCbgskgIypAtBRVtmFTnye0dYJd7vbpBInmLe0GRK9i8L15aMpO3I+65Sm5rlX4ZfSL1g4a+gLuwCM9ytOT+uKm1kQrywXV24WCavtCf8KQ2ip7SpB3bWSGvew0kwmQWsfodB7OxWAIxeuX7tjnntRFAcctTS9e0xNlxeHiMKjK3caXOsQzB1+l1Kk7p/UHxF6dOjQsMR9uS+WcINQFMXS2G6E2x4a8Y5XDRMTVzF+0gzhkbsM8NRLyJGifBh4NID+7smAPz3QQBO/kVgr07yloELQBS/Min/77u0TD9pPZDwvQAK6eOfB91YohQy2lhDYm4tunAyYzKUM1sAyCWvDCbI2cADyXCHTjpzNl6zjsQMJvWQVUfweA7u9NkzIsdhrwAMAqDNIz9xOx2pecDPwGX8fp2FveR6D78BIYBowg+TCLLMSeWmeJTlbHPQFT06NKlEaTaTUpzWYGkOHX3FedgP+Wy2wskJq4oJkdiQDHAU8mPASQBOhRa8oYgPJEZg0LDRPmsUfCckaQ/dRGY51bAxZy8KkEVehf0HR16is4RtnDvZg1lW9Kz/RoFE1UH/BkS3FJPlmjZKwfenI+timuw1OF7Yc23d2+e9MjhmfgXonSG6nMxtfBF54dlZ3Z876IvAiC6yNw2Uxqdl2SWAbTM2uI2SVJeL45qfDJzHCUKvOWySui/hd/L/nvZJwEQBoACa/un1WiG5aNEHINWzy1/ES5tF9RwsGMT1J/E2pWUBa6FD6q6gvtKA/vZvb4mEGjxP0EUxWuKpKjovva3QiRBLvLYZg6SGLyg+rtOdQotOV3AiO6NJQVWg/NMYbFaI2KUktPCQ41gEGiU7S4SK0zeoI4SVAtIArwUrkZO8XOq5xcULIQg/17TprWLrR0zA+BJ0rZp5FoPnINFE3EXlJDMzNrTo5r+9HlM2aa9ZSzwUtYzuk2HZJSyCpt0npmxlHMLIdXVeUmSk42j/qJt3d/OSzExdNPSO7FNz1ZLzmPwMYggmIya7JDHlEIZCvJ8a2P3gTdYOFGAeh++IiR2GbQAll06wxDITk5IterAEGgP2xbicnOgiYZVh+j0RGKvXq9LpFFScw+udN1sYsZdYktOpAMCa/oaJL0AHbGjfNEDaflydIkuGRPPr1cFKPH/p7IBeRTdl3F5FWAiaQbzQAHsKBBo6hTkA3+cqKjdQKGCN59jL5pdsJ8gWykFCgYa4lTatd2+Po+kRjsKeVtzKCpdpjBC0CDmjByd/poTv3o0NYwqosazA8Pk5ZGLacr2u5AEmCuIiYFBjhmq9u9DlXaK+K2dAjpZe1cJ1ll+cCHRAtUwwMrFFSKwMBTdEfyZTOnHvJXW4K0sYzcYT+ZHUq52QSxRrR3O7Ne0UMousHs1QRES7jAgChEJ7br+lUn6MCuqba7vo4ogBMOw5Ij1aR/fD2xMKoxIhtRlgD80EBZ4NsMON923b7uBuNwZrNH3k27Pyxzdx9MG29P1MkTHnLr5G9LtRsweVmGrJM9GksBgORvvB5xkP3m7U8OIlUlXGjYRh6tWUcYFJgX55UT9sEF02qa9NPMIBY4pQTs5xM1tFQM0CucErCcA8MRe4UfxqF5265t6oYZmJHQ/B1R5o97YTroX6ei80jzoN7GYNMW6RCkZjZSuQXTaBxUCkehDKZqPBjBrzpMBJWpRPNAifjxjQOsgkljUJqq8hes7Qva7ZqbtFoeUTpqFFvHsLX8HVB0wqZ0MNvq4fggQGcg221nthAiFzppCt74jAegZXCgjis4xylbhx4iaBEnlwVDO9CqoyAHy6QrrICbDIZMB48zEJkSoc1wSNE2BIBRsoQdd8bxCvwG2BtxwNVGscmAOeznZRNfow3d2ASoi+QW+mJQj/XmpcMe4w9ZrlyazPP+hJWQapNWKuvclka8ILPwQLwTdQzyjUg91P4WNxD9GH6ImCl7ztJZHBbZCXwGScGhMz7yU7BWVpQd+0j9oU8JuBbKIIEk4hDTEKoLybDYCvbiDEMiGmgUVwqpVj9TRt1tFO89uzem+WyTKh20F5iM4pnnwXXiqNMgIhBXgp2BjU3Rv8xTrGK+SQPfcHPIkB0TWFXUOyH0xxMm9BMjvBTvW1lmzLkIgMKYTD4ODsxXKWdhs1fdmrzw+pUPNYPXhtau6NVlDGUNxtO6+xgoLe9bnsn9np1RquWJYOGgCUYnVD+TNE7uZ4w2PUjbW+ZxmzBj4HuYttZKIcY99uEpO01QZ66pT9N2DhRyQIxRnQO4dRLgm4ckVyOKApj9GnPrZFo4UagUnXzMAQZc5SBnAGmp1co8Tw+wTTMX5VjN9nohbU0bVrPVsJrjn1++JJzmW8FkwBd9XcqbDr3dpObv4uIFkFN4cCj9xPC6y4dlmCvIjGmmFU/7zXUqaVXkioQgz60vlxDsQtphHIbeQOhAtatejNlyXtTmyCgPt5XMyBdi6XpVaMSvawNBA9Ekzedee79lrtFB9DF8ZN1R2NWPRRIp/yiVhzNEjXsOUn8LRk+WNRiUms00I+Dul1yzM4NJh7Ahu3mV3BkCuIv6k9tjKwNxsjTUnumM1+CsHQHBx3u6gjFFPHhWrZNRQoC6jhWY6oU2WtxGpAVyGgwNg04WmCq1iO1L6nof0i/TXA2oqGvXJoKckAN/bqrLYRnxXFy1Hz2U2qPNHsNHm1RVTIKTWDzhrKZVSCp/KeeqIlTOiCZMIYeKhe3aWs6LXMhEX9dgrci7YjWxeNQLYCwXaK+t9rZ6rp2IOqzIuhcbXyytQtKvpFnuLbHievUk2bYiDvTMTBy9yXZ9CdEgBQmyqM546101m5fodaw0f88mY6ObomsJgWyGTxQ6lVzEd4naDPQxkh9JdLoQc4b4Osx2AwTuEVhjp6WFaRQa5GwTRlyUAxTtktD8Jk6Ai6UnDLh76EfGLwGMuN1IJGGGrBo1sjTxGWyuBpBQNQdjE7jiG4arX1xWtAggNrSDLfmaJJq6uRDuHJnWw73jjbBodwtQM7VZY08LLMigHCqX1Rwk7iPsMnPO0dnRs6o674ZJRajsPEG0jIrAsnz87GaBwxgQwho+VP5Qyxhz1kPbNmtLaprLey7mjGSsD6uYG5om8Ccg3cgk6SHfxpMPBEjQ8e+vHtqqUjfMldrpJK4lVKN2pg+4MJdn1C7JE/BbqBcIBlfDDDXvazQ9j6OhyGIlO1SyrBCQRerE7nAddGaTW7tMVdcNV4olp2J6dtMnwrn5Eibcv2+z6i0AAn3cbQqw0g9U6VtClS4KWr3106FNJX+bAtjkexkHlQ+yTJZQfg3CWu2mWC/+B677CLaUyB0oJ0fiCdVl8ImVAgaPjXXuGGouMDpOpR6Tq815AtgBV/7sppzxJ8CIZTA1aGAfwjcE5gBQCIiX7B6ZvsALr7Q0aIPaMO9WAoGmBC/+PZXeyR6kBKShzC52A00BtaphDa+7tBMKnrIZyFM4N2aPXyR8HUQPBw0nRP5TIM+omu3w5NK11CFrhnWwYdGDe3JJRhX4dTNElFqw1jcrlRKCXyZe6xJ80KdoWwjSHV3S7FuBk02Wmoi9BNkCFmnWm0WoBa0G/irFQVSuZok5jC6iuzpFMr6poMf0jcGLIqGaTteAYQ0Ltn2yvL4wTMduqDn+FfVE9vlttaC7rgM/ZHsFPxH7HvKfjTtNQOU4E5RAblQkR95tMnVo7g7cF966aCJGTfgPBW7UeQEXOW8szxW/k0Ey3YhxbZclFSXAgwriZV4XHad299NP/pUGgIa2S6FiNvn2+Fi5+l4d/IMwilfQCHC/I11Et28AJ9Y9ZpBy3whxC7xlARoK8LKmRegXLnElW97Psgl2X8XdpkTcAsMeODuehF1LrMU+kh67vCQtlT1KFrOWRrGMztdiiQN9YSZRV7/a5lUWCKTHAUPNwM23PRi4JfjkaviV6z6gqIEA4Ip+An9GNT6KXrFSaIUm+/wniynpdlveYpuXfAqxP3KGBfYpIahMzWFLhSEitprm+EVBqxheThGS2akoYpA7l9irX8PZE9OqBSl8Mvw5qvIQ04l5KjtboSukLwwud8DOZMQSwFaMKz2rCyshlXStqyyHFNH3iBjMqqz4QVm2sIZ7ko096IgmCfM3OmkewNqZA7Q0kkzNSwDsYiv3vmkXAFuQ8iwGQcmSVVwNDEh3GhnJKkmbf/EUsfTIcm3lmGJ//kNSlwKG+vhdr0LjLxH7As5f/ThShZLZ3GOpHTII+lsgRT0n+lOolc10IDmiUx/HJNwvVtgJZN7kdlgBdjK303tVj1lJtdu+syK9Y4Y3gtLGU0FqtT/DmWDZMYXTGMWMOvpU4JkGTk8088DjSMXORO5TZdEopbR1Q37xzg0MQBxW2uUNDg2aIHBQKZq/TB2XpDFMfFhHOX+NiWGhfJOv3jwU3MyJGlRziN/3MbKmaYPSNvnuJcO6s5CAvie5YB1+uN/5u87njzBgyObz4XFgWSmnrIoRLAJhzwGz0S1fEthtwF3LWAYiagcw1ooqyLvXHnZCuEkThQppxwCIpiq4XwXHzWFmclPDwEZu3QT+bCZYmkE7uR3BAf8AS5esBWeYSi4IjgnEcplNgQCUa5tF78BeEbiwEl2OzbJZruftBdZPekPBfPjwx8tCL5vvRnemwiyP4pDrh8YaSS/F32hhmIX3lxsd6Rn0N4Gz0MStydLoGZAWnSpn7+i4tcWM3FcTDIQrBsfmlDwhdvkJwBdWm514He0L1lRFTIaJxmRRllPOrqXywqZUyNUpU15y2me36l2RGICPJRqsVrSq5NokBBGrlyslwYibQNVn3oQBN+rVGqXLqPE8q0IlACKdpGWSfUtU9461W+dy0PCfiu0c43BGdTOvsYW0krwAO7f4Sp1OiRp6TGEYJMV+bXGxGFwbXnoEeTRXpwLvokW7xhJ/amtDGVQaiq6bTnuLmA07Z9govsvRGTgbH5p5XySLYYEKnboDILtfZbSNhMJU9BcX4cYAzMohVyPjZwOv0jhbFTQK4hOf+elgIvnqYubtP85arp9VQ3XZD5GpaQqeBpFpdQICS7ANc9GDrIUhl7DQ+6Q4Jv2G8areYjBDyXaeROkkGUaHSxYR/3VjYO293yJwsj8opcQWnSQQ7XmrrTJ1URZenwcGYKjF13JqK1VjVxE81/RMKFAjZkCJq9oKrFHTNzEmG1XYCfd/JZ8iY6/EY0F6AU/DuelOALyrSvMhjR0VUNrFObLXVMnsobJgImkYxhRliJQGrorjTxJmom7C7PxiF6yS1ofnBxqjOeFnwuzdzeoZ2APoTYAnYi5zYYT5S5AxeLWjVvJpbHMlkMYBAaJGdcI6P04OEbuoy+QC6Eq6yBCwTQpUnRYEZ+1Km4B72KUGW9bDXz6kqGk+39JeCQWLJL7qyl0Q2ZXC3tRAiKYVVGsxJFs/1Mg4R3CDactEvxulQKx360PZTUakdYISh2Z7Dvni/h3mrFzmEww3ewMp5kAh/Ih6T2WYrXDR+C1+7Zyz2xcqn1Jod+vNd0T1+cHrWZrXE6bPSimzl6ifAqQXgry8TGIPhJ8TxAGllGwbiUcjVGnci325VFlZLwUp9/BiGsBFsnQtR86Rbo4dOk7uLEde5VU2RoXKkxJwsPhzIRtZwNqBTmTMy8oBSt1Eh9bPUOpWaG4y3EyHIhASDtmZ/hXA2tQjJALw2jLFiZVlxqUeNouH+ui33rLixkhqAnwTw6nBPOBfB6IWmqyFte9uygUGNRqVptHuXZZiHCdE6K1WBMo7CdjYKg43CTdtsWg5giP8/JIAaxtv59aL9wV5OjlAJ1e/QiUpLUNeK+cq99zZpXx0pZy/CSYkABvnp5BRbeW00REE69UtAxbC+wTOoZWhecFPyQCJGV8HFmbECme8wa4WoP9MR6ULkgMy5gXAnFd9kBtSV1nmcDw3I6iC3PQqevUsZea7G9/kBRF9UZzadVJ0Zuadb3pYGjNzG+J/aKcPec/m7nEoXKM78wY0/IQ9yNO63zAXYDsTTPjmusoW43PB8xs5XOvQDP4meR2po0/MQrUC8vSOXDLm6EOwQiwZkYWNk1ZnAoJ578O0YnWVqMGSdMdeg8PjIn16chhgs5FqaBcDiW1uOpuUiEy0LMoUTPehiVbejycGCyZClSdSmW/OtDisJU+wTWXVxzsBZBgZmR6HuALMjN7F291AhjbCw0SoOgJrgg2sJYtTJafM8BJvtSS1m15NO3AvCggkooKSwny0yXnfq1baCKLQ0S6JAy21+vGtdIVQ7fmra5vBzJhRM0NM4nRGMftnWjQ6enS4M6SuYp7IMMtLB39FTlM3z6ro2mqQzvjYuvmh9Oqj9NI87r5DpZeoaNWDq5b+mdZZmEP4W70KKy7+Vct2L9v0p7IgLGEzvJPqMC2r4mPICcO+cG7c29w5GlWlNlepjlXfOl1bqmSg4hJWG9jiYEHAx0053HLBOVhMzHZZvS66H4oVu1YVFjfRew7xbeEuRulgQ0o0RjGO/CEhzJXz/4RsUkRx4n2F0u+Ny+HWXafZuQz+CB97Hl4zP8hSsT2Hu+SKg9FRyOxXclY0ZK7jGlYycvm472ojKRNCHV8iZ77F2HR6/LxKcOB8/ZxcLotaF6mA1yYDAHhIHZtcpipm0FiVgcsBOEYCZReas0pmwkmuSH8JPUhbAkFvdStihpA4kDII/meiJeHCi7QwpHY5Ib/ouh3FoWlmj1tpjAlir5thQ9UmO4KFEt7QTdWsAIK3vno+hdYmjwCtL7owe5jA9Xsn9wg83JcKAnFqTwr1AYNk/u0lAUIpBFO8opOk6C7+82TEJhZlsjpYOTpQFzLT9DUAAh7qE1mCqZNGotJ64FCDF9mjHqklszN3v8QaioEfoQJPepeVdAYBdYnVd9snDKe+yvf72CxTEaId7oqxuKiL7tSfCFqhmCzHkeEe4DSCHkbFsvlbkBBotbfius6akQ/M9MFuZSQ4NxV277dzzB6WkIWpWdzMlJo3rlCby3bpTXswlAFb5ugeUkh4Q3NamkjhdS2B92Q9iwNL7aE9CtrOz+MVSDQu6nyBSuiuNBRKrPsV5FA6fZ5bIzeKinIDy/8vq5fWjYMYjyXWqnpIWez8xw9w0uDIoYuO497X6jGxVsJ6LOTd0Yjc1mGX2lfecE3OO6ky4Z0BnI6mzN+fd05QivvEansR3jmianfBkuQ+CiIYNysgztPb7mQuE3XTIHYV/lwmPAxUIiouvd6WYsBTgnAgYOO0i5lSZ9xlcvx9rV57wsdZlKrv6nDOtlRxhxW8ZxXPaRXxjuW/QBzhrB9kX+MtKcGjqFDzqEZQsR3QhH8RH0VdN5tMq4OT00u/wzHOI52dPcBp/iu73dBMnxcr9FRoxNiJyxFFLkU88h2shZD3Y0I3qyVojFRME0rXmqTqFsZ2OVxLXkYcuVJ3MTq37celU+05T42DGE2Cp3SxYvMx2dtKaFECJwceKO0zo2z7tdWj5qqrcz8TreI6mDBJqcR9XHYKrisaHsbGYQN6IJWFT7Y8jEWY2JDBTUOr5xIogjaCUmSswVw7zqrAtTbX1LOadmOTWTEUkg6Ep36FC9sfy9DUWG9NTCERK3gct2vwSr1YXkB20Q99Qu5ENXXYEiwhsYRBmj0EYpOCHZD9l29BQ1Yl4ui0DQ2dSXUV3a5IWVlqXTfRwtpt1C3IQiu2G0xJq3CWcqthNadKtaPCf48qWUwhQGFiLko2L8wkSB5wdpWvwHqyNS8rkScBGvgKq4RkbOslnIQlQY1qgtyZAjeVnI1VzKVOnjTSptfpwT1+1ejI3WCekiHisyR1ln0rvISyGo4V7VI4h2KhQ0xHpAASupvS1GxPeVksyaT32iLxItVvj6dv73oRgT97kU83HLMiCwu0M9ivujritQ7fl3ZWSy9rpS45WRGXmRWxScJIgPBKl7yFv6S61LkFvynUlYtBWcZmBjyqkD0Vu6zTKwFuSLRKo9KZil0JR1IdZqdLX1Bg+p3e1j3BpVrp1I3am1y1k9XbtcytRuxDHxEblZJlqcV5fJOo9RnYANDJoZUCeMiqaP3CE9fr4qW6U6WeB3hg4BWAqt8J8CD3xtggDwfpgP76ta/aDGDIBcQln6scSvaK2ZmjEOtmYc7c+rvhzG1emrIx+J+S19U6r1HdXC8FYPICdBGDTNWzLIBS6oLvFM2tE4CRxSMZMBnKm0KDjCYT7RR2pBQkjJCbVtfsCyHyZ/CrJYiBgCLi9TGsL6T7THkA63plMbCslLpuIuMnwcGyM5IuSA/s+RkRhRHnwFdV+K45mUvGQS9xF6CGgQ3dUkQbmpQT/QQtiWVamdVFuAuMdCPUW+Z8eptjBC+Q9kER/aCjtiFULC8DGTn/H+L9JWnB1Rr1IZBTufOyNfeifW5ro4NfifcYhVBlU79FYgxErWgMLy7X87qILuTjwBPVnmZyMacdMbxAcwN0cZM12v+JAnjI7ZuZ1pd5wgIVrRdUjeKUqWQ4wkYsri+pG43LIziYw6mmrqIAmlZv2cBkmpMG8sMCjF6bsoaGC4VCqNF7xm5UW2rUcD0gVNCKg64MNBnlbPS91CYJ9e4V18Gx9qJtwIp0oh8OYESRKdpOfCsOYOAaUJYLEeRj8XJ+OnJHbZCdIznpqE2tf7u4ZAHYTMAxiTcSocN1KPo3fPvxxUyJMCgtSCL1MqUpVfCPvLoBdhYOckD+Scgq7Q+40Dx3E44mD8KtfSSJst6b+JSlgNuWmmzbhKgPs2oOuyBZSiMAvmvc4k0mNFFkogRwlapzdxKHgu0SMgAO06K6MaM8v1MFSf1egvk/q5AEXKSluz5zo12tX8jypYhyQ+9dpEpSOCc4r7VbxIUjbzAqUQzZKoePdj/9NOVlWJKKXosVlc2xhgKiqpRFY6J+6yrBf/Tt3YJlRkBMpwJNEItVXKUqJvcMihQNugoqTEJjDN5DUXFQVSMrSwstMSgOyij6E1GXSCFiqLs+rLxE8/QxpNRvywIWWWXpyyxvkRz9BIVlw1ifIo3OolUUDJ971SeS3WsXlCJoTssPXnvGA39BM9JXXOa3j6pfkbSIarKGjvxxwn/84516eled2zfaObJv1G/iwtLQ6vGuiUBKNsksMeX5nx36CUB+d/vu9PIsFt8ZRtJ+m/0ksh8lDWf0DAXypmVJoswnqjkX/Vc4rva6lfkL0VOKX+nP8bEkZ4qLB2TDar4K7Sfzkz21Qkc2E2RBIoW3+XPuzBBm67quVCxlHr3QXXpUagBFWmRwb5Sd88YRJF+ogGFgDVItJrEdS4hzFJBkGpKkDt3Me4g2k88ugL3AQClNNzb9ulceuS9n+1rN2GffUasKxw469ju0TLyXOZguC/BMw6z1Q6mSGV3FcY27xFKunV5MmJnbRxToJCsDD7bx1skgMrNqWmQR0NUDhmT+n7VjGDyflDaFwYTw1i4aUtgGsKNBrrgsbM+4gtx5HSpm9BIGI2erWiRu1DhVRneKCahaYXCjD+J0Y9lXGo3G6RVn/qr8u0IPLhIZKrmIRST9wHxrc+4gUNeewu3yicAFkrlLTNRCHRF2AekBNSECoaElxcvvHqMJONYtjN0UqwDehkglruhfVYeavLT16tqSYMJh2uu23uQWR7T3xiGnqHToyjIJcVOx3JyNVBTn/KnCqAZI1QQ/QWKlyh8TxrOFq0Dr2JXgKPGdTgCYBgpyQGIXZYtBzWv7au6zFawSNGiIVzCMg+DySqqURWciw6AgoNiDpKzqoKo4T5xEfOF651QQiqsVN6wYpJlbJtzKy6gfSVbx3tPO5UrZWLD92SUBDTJYb9yssQa7OCter8VInIz98y0XYzptBHsayDOo8ND5Zkedp0ft0fBKXlUr7RCotbDisS22MCzxV7IcGqa2zQ3QB1aKt1YziM+s/T0RPGU+YufyZRxNWTKF0lTXlNyfENvaH/CJnhwTeMUZmyn+sQF49co3Rp8so4sEoTrPt8gPSiMvAx38rMwnReBphlpjZGTq9KAjLRc78f+rxdm6X2Zoe1eadBI06x6BkHBSkbm0QRFkmHmwEL4OBSi8krCzG2gv2ElwJJSyzUqzM4muWsmXIn/miMlZPWRsUOKYksGW/u5twv5E2r9i0klvdVuZO4wuIZcP36m0HQoOefOY9MBwa4/nK96TTSJSXwpGXawQ8WOcu2lCDWkz2OVg4q6dnYrsYfgtFaen2TyGO49HEjJ22x7bJ8S36ttEbQcy0TTAcHdJ1iEY25qHHZAjZ7SrVK7xviJLLJVNj+oyiPfFzLjJiA3OdRwS7Eduv5VhNoR4K++Vrg214j0NfuS7mexY8quwzAPwH1KKPrGZYud1dN8lEczio5P7VT0HbmEzhkbptV5Piau1XZ7H41v8xjs8t4LzcdGyaV/Z55WK1G58KRSqy0mnNeZJlNLitcFWTFkcE/rFppoo/DGRJlY7Ub+74jc/e1XMLiQ/2YqmCVF6AIxFc5IVBamqGdxGuRKGVvrnWFgzrFHDK//ixugWB/x7Ol0O7Xf5CEfmK2R1OFwvpjusJKqh+zdIKdRBdDamATxnYQcudHq2rNAIWNTOXkJVhgfLpVm2KZX5Ev6KnwN9GpsuMQ1sAGLixg0wlrh6KInou3aBSbZZzm3tMmgkdRgp2DG6SICF4vARcDYdPtL6ZVnPA8zl5EyGNnR4UNAz0FRb0lu66s3Wcr4S6sIdE8Vz7j6tfC1yJiiOmTQKuTFj9/gHz/3zOLywzf9wd/t6N62SM3w0vq8e2gH+Xuha4t/UCGpont7HI/tq1KCpoy1s8mg5pbo6flqqr8Ptk2i86/fYeHEcfzE6BE3DzabI9No+MdN1w4ZIL+e4hSMfEqlj0Q/k1bn5Fhv88P9etJI2bOMBAA=='
TRACE_ACTIONS = json.loads(gzip.decompress(base64.b64decode(TRACE_B64)).decode("utf-8"))

def _TRAN_BASE_AGENT(obs, config=None):
    step = min(int(obs.get("step", 0) or 0), len(TRACE_ACTIONS) - 1)
    return copy.deepcopy(TRACE_ACTIONS[step])


_TRAN_TERMINAL_PRODUCTS = {
    "WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON",
    "EGG", "MILK", "WOOL", "FERTILIZER",
}


def _best_terminal_item(inventory, prices):
    choices = [
        (int(prices.get(item, 0) or 0) * int(quantity or 0),
         int(prices.get(item, 0) or 0), int(quantity or 0), item)
        for item, quantity in (inventory or {}).items()
        if item in _TRAN_TERMINAL_PRODUCTS and int(quantity or 0) > 0
    ]
    return max(choices, default=None)


def _TRAN_TERMINAL_AGENT(obs, config=None):
    action = _TRAN_BASE_AGENT(obs, config)
    step = int(obs.get("step", 0) or 0)
    if step < 716:
        return action

    private = obs.get("private", {}) or {}
    inventories = private.get("inventories", []) or []
    shed = private.get("shed", {}) or {}
    prices = ((obs.get("market", {}) or {}).get("prices", {}) or {})
    player = int(obs.get("player", 0) or 0)
    farms = obs.get("farms", []) or []
    farm = farms[player] if 0 <= player < len(farms) else {}
    hand_count = len(farm.get("hands", []) or [])

    placed = {}
    farmer_choice = _best_terminal_item(
        inventories[0] if inventories else {}, prices
    )
    action["farmer"] = ["PASS"]
    if farmer_choice is not None:
        _, _, quantity, item = farmer_choice
        action["farmer"] = ["PLACE", item, quantity]
        placed[item] = placed.get(item, 0) + quantity

    hand_actions = []
    for index in range(hand_count):
        inventory = inventories[index + 1] if index + 1 < len(inventories) else {}
        choice = _best_terminal_item(inventory, prices)
        if choice is None:
            hand_actions.append(["PASS"])
            continue
        _, _, quantity, item = choice
        hand_actions.append(["PLACE", item, quantity])
        placed[item] = placed.get(item, 0) + quantity
    action["hands"] = hand_actions

    sale_totals = {
        item: int(shed.get(item, 0) or 0) + int(placed.get(item, 0) or 0)
        for item in _TRAN_TERMINAL_PRODUCTS
        if int(shed.get(item, 0) or 0) + int(placed.get(item, 0) or 0) > 0
    }
    ordered = sorted(
        sale_totals,
        key=lambda item: (
            int(prices.get(item, 0) or 0) * sale_totals[item],
            int(prices.get(item, 0) or 0),
            sale_totals[item],
            item,
        ),
        reverse=True,
    )
    action["market"] = [["SELL", item, sale_totals[item]] for item in ordered[:10]]
    return action


_TRAN_PENDING_PASTURE = None
_TRAN_FARMER_SHIFT_END = None


def _tran_tile_at(farm, position):
    if not isinstance(position, (list, tuple)) or len(position) != 2:
        return "OUT_OF_BOUNDS"
    column, row = map(int, position)
    tiles = farm.get("tiles", []) or []
    if not (0 <= row < len(tiles) and 0 <= column < len(tiles[row])):
        return "OUT_OF_BOUNDS"
    return tiles[row][column]


def agent(obs, config=None):
    global _TRAN_PENDING_PASTURE, _TRAN_FARMER_SHIFT_END

    step = int(obs.get("step", 0) or 0)
    if step == 0:
        _TRAN_PENDING_PASTURE = None
        _TRAN_FARMER_SHIFT_END = None
    action = copy.deepcopy(_TRAN_TERMINAL_AGENT(obs, config))
    if step >= 716:
        return action

    player = int(obs.get("player", 0) or 0)
    farms = obs.get("farms", []) or []
    farm = farms[player] if 0 <= player < len(farms) else None
    if not isinstance(farm, dict):
        return action

    hands = farm.get("hands", []) or []
    hand_actions = action.get("hands", []) or []

    if _TRAN_FARMER_SHIFT_END is not None:
        if step <= _TRAN_FARMER_SHIFT_END:
            previous = TRACE_ACTIONS[max(0, step - 1)] or {}
            action["farmer"] = copy.deepcopy(previous.get("farmer") or [])
        else:
            _TRAN_FARMER_SHIFT_END = None

    if _TRAN_PENDING_PASTURE is not None:
        channel, actor, position, expected_step = _TRAN_PENDING_PASTURE
        if step == expected_step:
            if channel == "farmer":
                current = farm.get("farmer")
                if list(current or []) == position and _tran_tile_at(farm, current) is None:
                    action["farmer"] = ["BUILD_PASTURE"]
            elif 0 <= actor < len(hands) and actor < len(hand_actions):
                if list(hands[actor]) == position and _tran_tile_at(farm, hands[actor]) is None:
                    hand_actions[actor] = ["BUILD_PASTURE"]
        _TRAN_PENDING_PASTURE = None

    farmer_position = farm.get("farmer")
    farmer_tile = _tran_tile_at(farm, farmer_position)
    if (
        action.get("farmer") == ["BUILD_PASTURE"]
        and isinstance(farmer_tile, dict)
        and farmer_tile.get("kind") == "WEED"
    ):
        action["farmer"] = ["DIG"]
        if step % 24 >= 20:
            _TRAN_FARMER_SHIFT_END = (step // 24 + 1) * 24 - 1
        _TRAN_PENDING_PASTURE = (
            "farmer", None, list(farmer_position), step + 1
        )

    for actor, requested in enumerate(hand_actions[: len(hands)]):
        if _TRAN_PENDING_PASTURE is not None:
            break
        if requested != ["BUILD_PASTURE"]:
            continue
        tile = _tran_tile_at(farm, hands[actor])
        if isinstance(tile, dict) and tile.get("kind") == "WEED":
            hand_actions[actor] = ["DIG"]
            _TRAN_PENDING_PASTURE = (
                "hands", actor, list(hands[actor]), step + 1
            )
            break
    action["hands"] = hand_actions
    return action


_SOIL_BASE_AGENT = agent

_SOIL_PRODUCTS = ('WOOL', 'MILK', 'STRAWBERRY', 'MELON')
_SOIL_HORIZON = 1
_SOIL_QUANTITY_FRACTION = 1.0
_SOIL_TRAILING_ONLY = False
_SOIL_CHECKPOINTS = (24, 48, 72, 96, 120, 144, 168)
_SOIL_PRIORITY_WEIGHT = {
    "WOOL": 3.2,
    "MILK": 2.0,
    "STRAWBERRY": 2.0,
    "MELON": 3.5,
}
_SOIL_CONFIDENCE = 0
_SOIL_REJECTED = False
_SOIL_LAST_STEP = -1


def _soil_get(obj, key, default=None):
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def _soil_iter_tiles(farm):
    for row in (_soil_get(farm, "tiles", []) or []):
        for tile in (row or []):
            if isinstance(tile, dict) or hasattr(tile, "kind"):
                yield tile


def _soil_public_signature(farm):
    counts = {
        "PASTURE": 0,
        "COOP": 0,
        "WEED": 0,
        "WHEAT": 0,
        "CARROT": 0,
        "TOMATO": 0,
        "STRAWBERRY": 0,
        "MELON": 0,
        "COW": 0,
        "SHEEP": 0,
        "GOOSE": 0,
    }

    for tile in _soil_iter_tiles(farm):
        animal = _soil_get(tile, "animal", None)
        crop = _soil_get(tile, "crop", None)
        kind = _soil_get(tile, "kind", None)

        if animal in counts:
            counts[animal] += 1
        elif crop in counts:
            counts[crop] += 1
        elif kind in counts:
            counts[kind] += 1

    farmer = tuple(_soil_get(farm, "farmer", []) or [])
    hands = tuple(
        tuple(position)
        for position in (_soil_get(farm, "hands", []) or [])
    )
    quadrants = tuple(
        _soil_get(farm, "unlocked_quadrants", []) or []
    )

    return (
        len(hands),
        quadrants,
        farmer,
        hands,
        tuple(counts[name] for name in sorted(counts)),
    )


def _soil_update_clone_detector(obs, step):
    global _SOIL_CONFIDENCE, _SOIL_REJECTED

    if _SOIL_REJECTED or step not in _SOIL_CHECKPOINTS:
        return

    farms = _soil_get(obs, "farms", []) or []
    player = int(_soil_get(obs, "player", 0) or 0)
    opponent = 1 - player

    if len(farms) < 2:
        _SOIL_REJECTED = True
        return

    own_farm = farms[player]
    opponent_farm = farms[opponent]

    same_topology = (
        _soil_public_signature(own_farm)
        == _soil_public_signature(opponent_farm)
    )
    own_money = float(_soil_get(own_farm, "money", 0) or 0)
    opponent_money = float(
        _soil_get(opponent_farm, "money", 0) or 0
    )
    same_money = abs(own_money - opponent_money) <= 1.0

    if same_topology and same_money:
        _SOIL_CONFIDENCE += 1
    else:
        _SOIL_REJECTED = True
        _SOIL_CONFIDENCE = 0


def _soil_clone_active():
    return (
        not _SOIL_REJECTED
        and _SOIL_CONFIDENCE >= 2
    )


def _soil_shed(obs):
    private = _soil_get(obs, "private", {}) or {}
    return _soil_get(private, "shed", {}) or {}


def _soil_prices(obs):
    market = _soil_get(obs, "market", {}) or {}
    return _soil_get(market, "prices", {}) or {}


def _soil_money_pair(obs):
    farms = _soil_get(obs, "farms", []) or []
    player = int(_soil_get(obs, "player", 0) or 0)
    opponent = 1 - player

    if len(farms) < 2:
        return 0.0, 0.0

    return (
        float(_soil_get(farms[player], "money", 0) or 0),
        float(_soil_get(farms[opponent], "money", 0) or 0),
    )


def _soil_existing_sales(orders):
    result = {}
    for order in orders:
        if (
            isinstance(order, list)
            and len(order) >= 3
            and order[0] == "SELL"
        ):
            item = order[1]
            result[item] = (
                result.get(item, 0)
                + max(0, int(order[2] or 0))
            )
    return result


def _soil_upcoming_sales(step):
    planned = {}
    end = min(
        len(TRACE_ACTIONS),
        step + _SOIL_HORIZON + 1,
    )

    for future_step in range(step + 1, end):
        distance = future_step - step
        for order in (
            TRACE_ACTIONS[future_step].get("market", []) or []
        ):
            if not (
                isinstance(order, list)
                and len(order) >= 3
                and order[0] == "SELL"
                and order[1] in _SOIL_PRODUCTS
            ):
                continue

            item = order[1]
            quantity = max(0, int(order[2] or 0))
            if item not in planned:
                planned[item] = [distance, quantity]
            else:
                planned[item][0] = min(
                    planned[item][0],
                    distance,
                )
                planned[item][1] += quantity

    return planned


def _soil_front_run(action, obs, step):
    if (
        step >= 716
        or _SOIL_HORIZON <= 0
        or not _soil_clone_active()
    ):
        return

    if _SOIL_TRAILING_ONLY:
        own_money, opponent_money = _soil_money_pair(obs)
        if own_money > opponent_money:
            return

    orders = list(action.get("market", []) or [])
    if len(orders) >= 10:
        return

    shed = _soil_shed(obs)
    prices = _soil_prices(obs)
    already_selling = _soil_existing_sales(orders)
    choices = []

    for item, (distance, planned_quantity) in (
        _soil_upcoming_sales(step).items()
    ):
        available = max(
            0,
            int(shed.get(item, 0) or 0)
            - already_selling.get(item, 0),
        )
        if available <= 0:
            continue

        quantity = min(available, planned_quantity)
        quantity = max(
            1,
            int(math.ceil(
                quantity * _SOIL_QUANTITY_FRACTION
            )),
        )
        quantity = min(quantity, available)

        price = float(prices.get(item, 0) or 0)
        priority = (
            price
            * quantity
            * _SOIL_PRIORITY_WEIGHT.get(item, 1.0)
            + (_SOIL_HORIZON + 1 - distance)
            * price
        )
        choices.append((priority, item, quantity))

    if not choices:
        return

    choices.sort(reverse=True)
    _, item, quantity = choices[0]
    orders.append(["SELL", item, quantity])
    action["market"] = orders[:10]


def agent(obs, config=None):
    global _SOIL_CONFIDENCE
    global _SOIL_REJECTED
    global _SOIL_LAST_STEP

    step = max(
        0,
        min(
            int(_soil_get(obs, "step", 0) or 0),
            len(TRACE_ACTIONS) - 1,
        ),
    )

    if step == 0 or step <= _SOIL_LAST_STEP:
        _SOIL_CONFIDENCE = 0
        _SOIL_REJECTED = False

    _SOIL_LAST_STEP = step
    _soil_update_clone_detector(obs, step)

    action = _SOIL_BASE_AGENT(obs, config)
    _soil_front_run(action, obs, step)
    return action
'''

agent_tree = ast.parse(AGENT_SOURCE)
compile(AGENT_SOURCE, "soil_v25_exact.py", "exec")
agent_namespace = {}
exec(AGENT_SOURCE, agent_namespace)
TRACE_ACTIONS = agent_namespace["TRACE_ACTIONS"]
assert len(TRACE_ACTIONS) == 720
assert agent_namespace["agent"]({"step": 0}) == TRACE_ACTIONS[0]
assert "#" not in AGENT_SOURCE

MAIN_PATH.write_text(AGENT_SOURCE, encoding="utf-8")
print({
    "trace_actions": len(TRACE_ACTIONS),
    "trace_sha256": hashlib.sha256(json.dumps(TRACE_ACTIONS, separators=(",", ":")).encode()).hexdigest(),
    "agent": "Soil V25 Exact",
})


## Replay Profile

In [ ]:
import matplotlib.pyplot as plt

activity = [
    sum(
        action != ['PASS']
        for name, group in step.items()
        for action in ([group] if name == 'farmer' else group)
    )
    for step in TRACE_ACTIONS
]
plt.figure(figsize=(12, 3.5))
plt.plot(activity, color='#f97316', linewidth=1.2)
plt.title('Replay activity by step')
plt.xlabel('Step')
plt.ylabel('Non-PASS actions')
plt.grid(alpha=0.25)
plt.show()


## Integrity

This submission is fixed to Soil V25 Exact. The checks verify its packed trace, source compilation and deterministic first action; no unavailable evaluator may silently replace it with another route.

In [ ]:
SELECTED_SOURCE = AGENT_SOURCE
SELECTED_NAME = "soil-v25-exact"

assert MAIN_PATH.read_text(encoding="utf-8") == AGENT_SOURCE
assert len(TRACE_ACTIONS) == 720
assert all(isinstance(action, dict) and {"farmer", "hands", "market"}.issubset(action) for action in TRACE_ACTIONS)
print({"selected": SELECTED_NAME, "integrity": "passed"})


## Package & Submit

In [ ]:
with tarfile.open(ARCHIVE_PATH, "w:gz") as archive:
    payload = SELECTED_SOURCE.encode("utf-8")
    info = tarfile.TarInfo("main.py")
    info.size = len(payload)
    archive.addfile(info, io.BytesIO(payload))

with tarfile.open(ARCHIVE_PATH, "r:gz") as archive:
    archived = archive.extractfile("main.py").read().decode("utf-8")

assert archived == SELECTED_SOURCE
print({
    "submission": str(ARCHIVE_PATH),
    "selected": SELECTED_NAME,
    "bytes": ARCHIVE_PATH.stat().st_size,
})
